In [28]:
! pip install nba_api pandas pyarrow

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
   - -------------------------------------- 1.0/27.3 MB 19.3 MB/s eta 0:00:02
   --------- ------------------------------ 6.8/27.3 MB 25.6 MB/s eta 0:00:01
   ------------------ --------------------- 12.6/27.3 MB 26.1 MB/s eta 0:00:01
   --------------------------- ------------ 18.6/27.3 MB 26.8 MB/s eta 0:00:01
   ---------------------------------- ----- 23.9/27.3 MB 26.7 MB/s eta 0:00:01
   ---------------------------------------- 27.3/27.3 MB 26.1 MB/s  0:00:01



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\yaman\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import nba_api
from nba_api.stats.static import teams, players
from nba_api.live.nba.endpoints import scoreboard
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.endpoints import playbyplayv2, playbyplayv3
import pandas as pd

In [3]:
def pull_live_data():
    try:
        scoreboard_data = scoreboard.ScoreBoard().games.get_dict()['scoreboard']
        return pd.DataFrame(scoreboard_data)
    except Exception as e:
        print(f"Error pulling live data: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in case of error

Preprocessing of live data pulled to better understand structure

In [4]:
data = pull_live_data()

Error pulling live data: Expecting value: line 1 column 1 (char 0)


In [5]:
def pull_play_by_play_data(game_id):   
    play_by_play_v3 = playbyplayv3.PlayByPlayV3(game_id=game_id).get_data_frames()[0]
    return play_by_play_v3

In [6]:
def get_team_id(team_name):
    nba_teams = teams.get_teams()
    team_info = [team for team in nba_teams if team["abbreviation"] == team_name][0]
    return team_info['id']

In [7]:
gsw_id = get_team_id("GSW")
lal_id = get_team_id("LAL")

In [8]:
from nba_api.stats.endpoints import leaguegamefinder

gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=gsw_id)
# The first DataFrame of those returned is what we want.
games = gamefinder.get_data_frames()[0]
games.head()

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [9]:
pull_play_by_play_data('0052500131').head()

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

Combine Data from play-by-play data for training neural net, pull from 25-26 season 

In [11]:
all_games = leaguegamefinder.LeagueGameFinder().get_data_frames()[0]
all_games.shape

(30000, 28)

In [12]:
all_games.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,42025,1610612752,NYK,New York Knicks,0042500303,2026-05-23,NYK @ CLE,W,240,121,...,0.889,8,29,37,27,11,4,14,19,13.0
1,42025,1610612739,CLE,Cleveland Cavaliers,0042500303,2026-05-23,CLE vs. NYK,L,241,108,...,0.632,9,25,34,22,7,1,17,22,-13.0
2,42025,1610612759,SAS,San Antonio Spurs,0042500313,2026-05-22,SAS vs. OKC,L,241,108,...,0.818,9,28,37,26,8,6,15,28,-15.0
3,42025,1610612760,OKC,Oklahoma City Thunder,0042500313,2026-05-22,OKC @ SAS,W,241,123,...,0.848,7,34,41,29,7,2,10,25,15.0
4,42025,1610612739,CLE,Cleveland Cavaliers,0042500302,2026-05-21,CLE @ NYK,L,240,93,...,0.688,13,29,42,15,4,5,8,20,-16.0


In [ ]:
all_games['SEASON_ID'].unique()

<StringArray>
['42025', '52025', '22025', '32025', '62025', '12025', '42024', '52024',
 '22024', '32024', '62024', '12024', '42023', '52023', '22023', '32023',
 '62023', '12023', '42022', '52022', '22022', '32022', '12022', '42021',
 '52021', '22021', '32021', '12021', '42020', '52020', '22020', '32020',
 '12020', '42019', '52019', '22019', '12019', '32019', '42018', '22018',
 '32018', '12018', '42017', '22017', '32017', '12017', '42016', '22016',
 '32016', '12016', '42015', '22015', '32015', '12015']
Length: 54, dtype: str

In [ ]:
all_games[all_games['SEASON_ID'] == '52025']

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
150,52025,1610612744,GSW,Golden State Warriors,0052500211,2026-04-17,GSW @ PHX,L,240,96,...,0.826,12,27,39,22,5,0,20,19,-15.0
151,52025,1610612766,CHA,Charlotte Hornets,0052500201,2026-04-17,CHA @ ORL,L,242,90,...,0.880,7,27,34,17,7,4,16,33,-31.0
152,52025,1610612756,PHX,Phoenix Suns,0052500211,2026-04-17,PHX vs. GSW,W,240,111,...,1.000,9,27,36,23,14,4,12,23,15.0
153,52025,1610612753,ORL,Orlando Magic,0052500201,2026-04-17,ORL vs. CHA,W,242,121,...,0.784,11,38,49,27,6,8,14,31,31.0
154,52025,1610612746,LAC,LA Clippers,0052500131,2026-04-15,LAC vs. GSW,L,240,121,...,1.000,9,27,36,30,15,3,17,23,-5.0
155,52025,1610612744,GSW,Golden State Warriors,0052500131,2026-04-15,GSW @ LAC,W,241,126,...,0.619,10,26,36,29,12,3,20,19,5.0
156,52025,1610612755,PHI,Philadelphia 76ers,0052500101,2026-04-15,PHI vs. ORL,W,239,109,...,0.840,7,33,40,17,8,9,11,27,12.0
157,52025,1610612753,ORL,Orlando Magic,0052500101,2026-04-15,ORL @ PHI,L,240,97,...,0.774,9,32,41,19,5,4,14,21,-12.0
158,52025,1610612757,POR,Portland Trail Blazers,0052500121,2026-04-14,POR @ PHX,W,239,114,...,0.739,13,29,42,27,9,10,15,27,4.0
159,52025,1610612766,CHA,Charlotte Hornets,0052500111,2026-04-14,CHA vs. MIA,W,265,127,...,1.000,17,37,54,26,4,7,6,21,1.0


In [15]:
all_games[all_games['SEASON_ID'] == '22025']

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
162,22025,1610612754,IND,Indiana Pacers,0022501188,2026-04-12,IND vs. DET,L,239,121,...,0.800,8,23,31,31,15,6,16,17,-12.0
163,22025,1610612741,CHI,Chicago Bulls,0022501193,2026-04-12,CHI @ DAL,L,241,128,...,0.737,13,29,42,30,12,3,13,20,-21.0
164,22025,1610612740,NOP,New Orleans Pelicans,0022501195,2026-04-12,NOP @ MIN,L,240,126,...,0.816,20,32,52,20,9,6,9,28,-6.0
165,22025,1610612766,CHA,Charlotte Hornets,0022501190,2026-04-12,CHA @ NYK,W,238,110,...,0.765,12,34,46,27,7,7,17,13,14.0
166,22025,1610612749,MIL,Milwaukee Bucks,0022501191,2026-04-12,MIL @ PHI,L,239,106,...,0.682,12,33,45,26,8,2,17,15,-20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2635,22025,1610612750,MIN,Minnesota Timberwolves,0022500089,2025-10-22,MIN @ POR,W,239,118,...,0.792,10,35,45,21,10,10,19,29,4.0
2636,22025,1610612760,OKC,Oklahoma City Thunder,0022500001,2025-10-21,OKC vs. HOU,W,290,125,...,0.800,11,27,38,29,12,4,11,27,1.0
2637,22025,1610612747,LAL,Los Angeles Lakers,0022500002,2025-10-21,LAL vs. GSW,L,240,109,...,0.607,7,32,39,23,7,2,19,21,-10.0
2638,22025,1610612745,HOU,Houston Rockets,0022500001,2025-10-21,HOU @ OKC,L,292,124,...,0.871,16,36,52,23,6,5,22,26,-1.0


In [20]:
recent_season_game_ids = all_games.iloc[0:5000]['GAME_ID'].tolist() # most recent 5000 games as training data

In [21]:
# export recent season game ids to csv for later use 
pd.DataFrame(recent_season_game_ids, columns=['GAME_ID']).to_csv('../data/recent_season_game_ids.csv', index=False)

In [ ]:
import time
import random
from requests.exceptions import Timeout, ConnectionError
import os

def pull_play_by_play_data_rate_limited(game_id, retries=3):
    for attempt in range(retries):
        try:
            time.sleep(random.uniform(0.6, 1.2))  # rate limit between calls
            pbp = playbyplayv3.PlayByPlayV3(game_id=game_id, timeout=30)
            return pbp.get_data_frames()[0]
        except (Timeout, ConnectionError, Exception) as e:
            print(f"Attempt {attempt+1} failed for {game_id}: {e}")
            time.sleep(2 ** attempt)  # exponential backoff: 1s, 2s, 4s
    print(f"Skipping {game_id} after {retries} attempts")
    return pd.DataFrame()

In [32]:
def combine_play_by_play(game_ids, checkpoint_dir='checkpoints/'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    combined = pd.DataFrame()
    
    for i, game_id in enumerate(game_ids):
        cache_path = f"{checkpoint_dir}{game_id}.csv"
        if os.path.exists(cache_path):
            pbp = pd.read_csv(cache_path)
        else:
            pbp = pull_play_by_play_data(game_id)
            if not pbp.empty:
                pbp.to_csv(cache_path, index=False)
        
        if not pbp.empty:
            combined = pd.concat([combined, pbp], ignore_index=True)
        print(f"{i+1}/{len(game_ids)} done")
    
    return combined

In [33]:
combined_play_by_play_data = combine_play_by_play(recent_season_game_ids)

1/5000 done
2/5000 done
3/5000 done
4/5000 done
5/5000 done
6/5000 done
7/5000 done
8/5000 done
9/5000 done
10/5000 done
11/5000 done
12/5000 done
13/5000 done
14/5000 done
15/5000 done
16/5000 done
17/5000 done
18/5000 done
19/5000 done
20/5000 done
21/5000 done
22/5000 done
23/5000 done
24/5000 done
25/5000 done
26/5000 done
27/5000 done
28/5000 done
29/5000 done
30/5000 done
31/5000 done
32/5000 done
33/5000 done
34/5000 done
35/5000 done
36/5000 done
37/5000 done
38/5000 done
39/5000 done


ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)